In [18]:

import streamlit as st
from ollama import Client
import json
import os

# Default settings if file not found
DEFAULT_SETTINGS = {
    # "model": "llama3.1:latest",
    "model": "llama3.1:70b",
    "temperature": 0.0,
    "system_prompt": """You are an advanced AI designed to analyze clinical notes of mental health patients, leveraging your understanding of psychological frameworks and therapeutic models, including the "Five Ps" framework. Your task is to identify, count, and explain occurrences of six key factors in the text. These factors may be explicitly stated or implied through context, requiring your advanced understanding of mental health concepts and terminology. 
"input texts are messy with html. Please carefullyt consider the text"
Key Responsibilities:
1. **Identify Factors**:
   - Analyze the text for mentions or implications of the six factors: presenting, predisposing, precipitating, perpetuating, integrated, and protective. 
   - Use your knowledge to interpret context, avoiding reliance solely on specific keywords.

2. **Interpret Context**:
   - Consider the patient's history, environmental factors, relationships, coping mechanisms, and current circumstances to deduce the presence of factors.
   - DO NOT Recognize implied factors based on holistic descriptions, such as "support from family" THAT CAOULD imply protective factors BUT ASK THE QUESTION "COULD THIS BE A PROTECTIVE FACTOR. LIKEWISE WITH A  "stressful work environment" DO NOT imply perpetuating or precipitating factors. THE REALITY MAY CONTRADICT USUAL REASONING

3. **Count Occurrences**:
   - Each mention or implication of a factor should be counted separately.
   - For example, mentions of "family support" and "supportive friendships" should be counted as two occurrences of protective factors. IMPLICATION OF A FACTOR SHOULD BE DERIVED FROM A SPECIFIC KEYWORD SUCH AS SUPPORTIVE WHERE THERE IS NO POTENTIAL FOR MISINTERPRETATION.

4. **Provide Explanations**:
   - For each factor detected, think an explanation in bullet points that outlines:
     - The specific part(s) of the text where the factor was found.
     - Why it qualifies as that factor based on context and meaning. Write this in detail. Why it falls into this category and how this text supports it.

5. **Avoid Hallucination**:
   - Only report factors that are directly supported by the text or clearly inferred from the context.
   - Do not introduce factors or details not evident in the provided information.

6. **Output Requirements**:
   - Return the results as a JSON object structured as:
     {
       "integrated": {"count": <integer>},
       "presentation": {"count": <integer>, },
       "predisposing": {"count": <integer>, },
       "precipitating": {"count": <integer>, },
       "perpetuating": {"count": <integer>, },
       "protective": {"count": <integer>, }
     }


8.most important:

Key Factors to Detect:
1. **Integrated Factors**:
   - These factors provide a cohesive understanding of the patient’s overall goals, diagnosis, risks, or available resources. Examples include mentions of a diagnosis, treatment plan, or systemic risks. risk state and risk status, foreseeable falls in this category.

2.  **Presentation Factors**:
   - These factors describe how the patient’s condition manifests, such as diagnosis, symptoms, presenting problems, or recent episodes (e.g. "ongoing anxiety," "panic attacks", concerns, current episode, experiencing, clinical "history of")). This goes beyond diagnosis to include what the person and clinician identify as difficulties, how the person’s life is affected, and when a particular difficulty should be targeted for intervention. For example, while a person may meet criteria for the diagnosis of borderline personality disorder, presenting difficulties may include not being able to maintain employment, erratic friendships, and physical health complications resulting from self-harm. Specifying such difficulties can allow for a more focused intervention.

3. **Predisposing Factors**:
   - Historical or hereditary aspects that make the patient vulnerable to mental health challenges. This comprises identifying possible biological contributors (for example, organic brain injury and birth difficulties), genetic vulnerabilities (including family history of mental health difficulties), environmental factors (such as socio-economic status, trauma, or attachment history) and psychological or personality factors (including core beliefs or personality factors) which may put a person at risk of developing a specific mental health difficulty. They may be be biological (e.g. genetic, birth trauma, brain injury, psychiatric illness, physical illness, medication, drugs, alcohol, pain) or psychological (e.g. personality, modelling, uncounscious defences, conscious coping strategies, self-esteem, body image, cognition) or social (e.g. socio-economic status, trauma).

4. **Precipitating Factors**:
   - These are triggers or immediate events causing the patient’s condition to worsen (e.g., "work triggers," "recent stress"). This can include significant events preceding the onset of the disorder, such as substance use, or interpersonal, legal, occupational, physical, or financial stressors. They may be be biological (e.g. medication, trauma, drugs, alcohol, acute illness, pain) or psychological (e.g. stage of life, loss, grief, treatment, stressors) or social (e.g. work, finances, connections, relationships).

5. **Perpetuating Factors**:
   - Ongoing conditions or behaviors sustaining the patient’s difficulties (e.g., "conflict at home," "regular stressors"). This comprises factors which maintain the current difficulties. These can include ongoing substance use, repeating behavioral patterns (including avoidance or safety behaviors in anxiety disorders, or withdrawal in depressive disorders), biological patterns (such as insomnia in mania, and insomnia or hypersomnia in depression) or cognitive patterns such as attentional biases, memory biases, or hypervigilance.

6. **Protective Factors**:
   - Positive aspects that mitigate challenges, such as supportive relationships, personal strengths, or effective coping strategies (e.g., "supportive friends," "strong family system"). This involves identifying strengths or supports that may mitigate the impact of the disorder. These can include social support, skills, interests, and some personal characteristics. 

Guidelines for Analysis:
- Apply your expertise in mental health concepts and therapeutic practices to ensure a thorough and accurate analysis.
- Use your contextual understanding to bridge gaps between explicit mentions and implied meanings.
- Be comprehensive in your analysis, ensuring no factor is overlooked if supported by the text.
- Protective factors are particularly important element which has traditionally been lacking in mental health interventions, but inclusion of which results in a higher likelihood of reduced symptomatology and increased resilience. We would add that identification of protective factors also creates increased optimism in both the clinician and patient and contributes to a positive therapeutic relationship.
"""
}

# SETTINGS_FILE = "system_settings.json"

# def load_settings():
#     if not os.path.exists(SETTINGS_FILE):
#         with open(SETTINGS_FILE, 'w') as f:
#             json.dump(DEFAULT_SETTINGS, f, indent=4)
#     with open(SETTINGS_FILE, 'r') as f:
#         return json.load(f)

# def save_settings(settings):
#     with open(SETTINGS_FILE, 'w') as f:
#         json.dump(settings, f, indent=4)

# # Load current settings
settings = DEFAULT_SETTINGS

# Initialize the Ollama client
client = Client(
    host='http://localhost:11434',
    headers={'x-some-header': 'some-value'}
)

def analyze_clinical_notes_without_explanations(text, temperature=0):
    # Define the payload for the API request
    payload = {
        "model": settings["model"],
        "prompt": text,
        "stream": False,
        "format": {
            "type": "object",
            "properties": {
                "integrated": {
                    "type": "object",
                    "properties": {
                        "count": {"type": "integer"},
                        # "explanations": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["count",]
                },
                "presentation": {
                    "type": "object",
                    "properties": {
                        "count": {"type": "integer"},
                        # "explanations": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["count",]
                },
                "precipitating": {
                    "type": "object",
                    "properties": {
                        "count": {"type": "integer"},
                        # "explanations": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["count", ]
                },
                "predisposing": {
                    "type": "object",
                    "properties": {
                        "count": {"type": "integer"},
                        # "explanations": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["count", ]
                },
                "perpetuating": {
                    "type": "object",
                    "properties": {
                        "count": {"type": "integer"},
                        # "explanations": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["count", ]
                },
                "protective": {
                    "type": "object",
                    "properties": {
                        "count": {"type": "integer"},
                        # "explanations": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": ["count",]
                }
            },
            "required": ["integrated", "presentation", "precipitating", "predisposing", "perpetuating", "protective"]
        },
        "options": {
            "temperature": temperature,
            "num_ctx": 40000
        },
        "system": settings["system_prompt"]
    }

    # Send the request to the model
    response = client.generate(**payload)

    # Extract and return the JSON response
    try:
        return json.loads(response.response)
    except json.JSONDecodeError:
        return {"error": "Invalid response format"}

In [19]:
test_text = "I am in safe space, presentation:  found"
analyze_clinical_notes_without_explanations(test_text, 0)

{'integrated': {'count': 0},
 'presentation': {'count': 1},
 'precipitating': {'count': 0},
 'predisposing': {'count': 0},
 'perpetuating': {'count': 0},
 'protective': {'count': 1}}

In [20]:
df.shape

(2330516, 15)

In [5]:
import pandas as pd
from collections import Counter
import numpy as np


In [6]:
data_path = "/home/knhuq/work/databse_final/data.csv"
df = pd.read_csv(data_path)
all_facility_w_count = dict(Counter(list(df['encounterFacility'])))
all_keys = list(filter(bool,list(all_facility_w_count.keys())))
all_keys = [i for i in all_keys if i is not np.nan]



prelist = [i for i in all_keys if 'royal' in i.lower().strip()] + [i for i in all_keys if 'north' in i.lower().strip()]
other_list = [i[0] for i in sorted([(k,v) for k,v in all_facility_w_count.items()],key = lambda x:x[1], reverse = True) if i[1] > 1000]
other_list = [i for i in other_list if i not in prelist ]
other_list = [i for i in other_list if i is not np.nan]
final_list = prelist + other_list

In [22]:
all_index = list(df.index)
all_encounter_facility = list(df['encounterFacility'])
encounter_facility_zipped = list(zip(all_encounter_facility, all_index))
index_map = {k: i for i, k in enumerate(final_list)}
# Sort x based on the sequence from l
encounter_facility_zipped_sorted = sorted(encounter_facility_zipped, key=lambda item: index_map.get(item[0], float('inf')))
encounter_facility_zipped_sorted[5000:6000]

[("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274438),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274449),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274529),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274559),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274626),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274663),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274682),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274902),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274957),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274971),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274984),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 274990),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 275112),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 275148),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 275200),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 275205),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 275334),
 ("ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL", 2

In [8]:
df = df.dropna(subset=['progressNote'])

In [9]:
checkpoint_path = os.path.join(SAVING_PATH,'checkpoint.txt')
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'r') as f:
        lines = f.readlines()

NameError: name 'SAVING_PATH' is not defined

125

In [10]:
import os
from tqdm import tqdm
SAVING_PATH = "/home/knhuq/work/database_extracted_chunks"
SAVING_THRESHOLD = 1000


checkpoint_path = os.path.join(SAVING_PATH,'checkpoint.txt')
try:
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r') as f:
            lines = f.readlines()
            START_POINT = int(lines[0].split(',')[-1])
    else:
        START_POINT = 0
except:
    START_POINT = 0
    
print (f'Starting from {START_POINT}')

list_of_current_rows = []
list_of_current_extracted_values = []
CURRENT_ITERATION = 0
for ef,row_indx in tqdm(encounter_facility_zipped_sorted[START_POINT:]):
    CURRENT_ITERATION = CURRENT_ITERATION + 1
    START_POINT = START_POINT + 1
    current_row = df.iloc[row_indx]
    extracted_values = analyze_clinical_notes_without_explanations(current_row['progressNote'])
    list_of_current_extracted_values.append(extracted_values)
    list_of_current_rows.append(current_row)
    
    if CURRENT_ITERATION %  SAVING_THRESHOLD == 0 and CURRENT_ITERATION !=0:
        current_df = pd.DataFrame(list_of_current_rows)
        current_df['extracted_values'] = list_of_current_extracted_values
        current_df.to_csv(os.path.join(SAVING_PATH, f'{START_POINT}.csv'))
        
        with open(checkpoint_path, 'w') as f:
             f.write(f'{ef},{row_indx},{START_POINT}')
        
        list_of_current_rows = []
        list_of_current_extracted_values = []
        CURRENT_ITERATION = 0

    
    

Starting from 1000


  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        | 16/2970010 [01:25<4390:05:08,  5.32s/it]


KeyboardInterrupt: 

In [188]:
lines
   

[]

In [167]:
os.path.join(SAVING_PATH, f'{START_POINT}.csv')

'work/database_extracted_chunks/1.csv'

In [168]:
!pwd

/home/knhuq/t25/healthai/notebooks


In [111]:
iteration_saving_threshold = 2000
global_iter = 0

for encounter_facility in final_list:
    df_for_current_ef = df[df['encounterFacility'] == encounter_facility]
    df_for_current_ef = df_for_current_ef.dropna(subset=['progressNote'])
    df_for_current_ef['extracted_values'] = ''
    extracted_values_col = []
    for itr, progress_note in enumerate(list(df_for_current_ef['progressNote'])):
        extracted_values = analyze_clinical_notes_without_explanations(progress_note)
        extracted_values_col.append(extracted_values)
        global_iter = global_iter + 1
        

/tmp/ipykernel_50032/1098975721.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[row_indx]['a'] = 1
/tmp/ipykernel_50032/1098975721.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[row_indx]['a'] = 1


2971010

In [123]:
iteration_saving_threshold = 2000
last_encounter_facility = None
last_chunk_number = 0

try:
    # Try to load last encounter facility and chunk number from file
    with open('last_checkpoint.txt', 'r') as f:
        lines = [line.strip() for line in f.readlines()]
        if len(lines) == 2:
            last_encounter_facility = lines[0]
            last_chunk_number = int(lines[1])
except FileNotFoundError:
    # File not found, this is the first run
    pass

chunk_start_index = iteration_saving_threshold * (last_chunk_number - 1)
global_iter = chunk_start_index + 1

for encounter_facility in final_list:
    if encounter_facility == last_encounter_facility and last_encounter_facility is not None:
        # If this is a restart, skip rows that were already processed
        df_for_current_ef = df[df['encounterFacility'] == encounter_facility].iloc[chunk_start_index:]
    else:
        df_for_current_ef = df[df['encounterFacility'] == encounter_facility]
    
    df_for_current_ef = df_for_current_ef.dropna(subset=['progressNote'])
    
    # Initialize extracted values column
    df_for_current_ef['extracted_values'] = ''
    chunk_number = last_chunk_number
    
    while True:
       
        chunk_end_index = min(chunk_start_index + iteration_saving_threshold, len(df_for_current_ef))
        
        for itr, progress_note in enumerate(list(df_for_current_ef['progressNote'].iloc[:chunk_end_index])):
            print (f'{chunk_start_index}/{len(df)}', end ='\r')
            extracted_value = analyze_clinical_notes_without_explanations(progress_note)
            df_for_current_ef.loc[(df_for_current_ef.index == df_for_current_ef.index[itr]), 'extracted_values'] = extracted_value
            global_iter += 1
        
        # Save the current chunk to a CSV file
        df_for_current_ef.iloc[:chunk_end_index].to_csv(f"chunk_{chunk_number}.csv", index=False)
        
        if chunk_end_index >= len(df_for_current_ef):
            break
        
        chunk_start_index = chunk_end_index + iteration_saving_threshold
        chunk_number += 1
    
    # Save current encounter facility and chunk number to file for next run
    with open('last_checkpoint.txt', 'w') as f:
        f.write(encounter_facility + '\n')
        f.write(str(chunk_number) + '\n')

    print(f"Completed processing for {encounter_facility}")

KeyboardInterrupt: 

In [116]:
# Define constants
ITERATION_SAVING_THRESHOLD = 2000

try:
    # Try to load last encounter facility and chunk number from file
    with open('last_checkpoint.txt', 'r') as f:
        lines = [line.strip() for line in f.readlines()]
        if len(lines) == 2:
            LAST_ENCOUNTER_FACILITY = lines[0]
            LAST_CHUNK_NUMBER = int(lines[1])
except FileNotFoundError:
    # File not found, this is the first run
    LAST_ENCOUNTER_FACILITY = None
    LAST_CHUNK_NUMBER = 0

# Calculate chunk start index
CHUNK_START_INDEX = ITERATION_SAVING_THRESHOLD * (LAST_CHUNK_NUMBER - 1)
GLOBAL_ITER = CHUNK_START_INDEX + 1

def process_encounter_facility(encounter_facility, df):
    # Initialize extracted values column
    df['extracted_values'] = ''

    chunk_number = LAST_CHUNK_NUMBER
    chunk_start_index = CHUNK_START_INDEX

    while True:
        chunk_end_index = min(chunk_start_index + ITERATION_SAVING_THRESHOLD, len(df))
        
        for itr in tqdm(range(chunk_end_index), desc=f"Processing {encounter_facility}"):
            progress_note = df['progressNote'].iloc[itr]
            extracted_value = analyze_clinical_notes_without_explanations(progress_note)
            df.loc[df.index[itr], 'extracted_values'] = extracted_value
            global GLOBAL_ITER
            GLOBAL_ITER += 1
        
        # Save the current chunk to a CSV file
        df.iloc[:chunk_end_index].to_csv(f"chunk_{chunk_number}.csv", index=False)
        
        if chunk_end_index >= len(df):
            break
        
        chunk_start_index = chunk_end_index + ITERATION_SAVING_THRESHOLD
        chunk_number += 1
    
    # Save current encounter facility and chunk number to file for next run
    with open('last_checkpoint.txt', 'w') as f:
        f.write(encounter_facility + '\n')
        f.write(str(chunk_number) + '\n')

def main():
    total_samples = len(df)
    start_time = time.time()

    for encounter_facility in final_list:
        if encounter_facility == LAST_ENCOUNTER_FACILITY and LAST_ENCOUNTER_FACILITY is not None:
            # If this is a restart, skip rows that were already processed
            df_for_current_ef = df[df['encounterFacility'] == encounter_facility].iloc[CHUNK_START_INDEX:]
        else:
            df_for_current_ef = df[df['encounterFacility'] == encounter_facility]
        
        df_for_current_ef = df_for_current_ef.dropna(subset=['progressNote'])
        process_encounter_facility(encounter_facility, df_for_current_ef)
    
    end_time = time.time()
    print(f"Processing completed in {end_time - start_time} seconds")

In [117]:
main()

Processing ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL: 0it [00:00, ?it/s]
Processing ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   | 0/4000 [00:03<?, ?it/s]


ValueError: Incompatible indexer with Series

In [1]:
from glob import glob

In [5]:
chunks_path = '/home/knhuq/work/database_final_chunks/*.csv'

In [6]:
files = glob(chunks_path)

In [9]:
import pandas as pd
list_of_df = [pd.read_csv(i) for i in files]

In [13]:
df = pd.concat(list_of_df)

In [15]:
df.head()

,Unnamed: 0.2,index,Unnamed: 0.1,Unnamed: 0,FormID,ConsumerID,TemplateName,clinicalNoteDate,clinicalNoteTime,encounterFacility,consumerEncounterUrNumber,consumerId,consumerSurname,consumerGivenNames,consumerFullLivingAddress,consumerDob,consumerSex,progressNote,extracted_values
0,1688868,2179099.0,2272324.0,2272324,42168799.0,635057.0,Progress Note,01/10/2019,09:00,INNER NORTH BRISBANE COMMUNITY MHS,3241737,635057.0,PATERSON,Damien,Ozcare Men’s Hostel 48 Peel St South Brisbane...,04/09/1992,Male,&lt;strong&gt;Situation&lt;/strong&gt;:&lt;br&...,"{'integrated': {'count': 0}, 'presentation': {..."
1,1688876,2179111.0,2272336.0,2272336,42168665.0,490368.0,Progress Note,01/10/2019,11:15,INNER NORTH BRISBANE COMMUNITY MHS,3303915,490368.0,MEKOYA,DERSEH,325 Beaudesert Rd Moorooka QLD 4105,26/08/1991,Male,"&lt;br&gt;Phone call to Des p. 0410929345 , ni...","{'integrated': {'count': 0}, 'presentation': {..."
2,1688887,2179128.0,2272353.0,2272353,42168682.0,635057.0,Progress Note,01/10/2019,11:26,INNER NORTH BRISBANE COMMUNITY MHS,3241737,635057.0,PATERSON,Damien,Ozcare Men’s Hostel 48 Peel St South Brisbane...,04/09/1992,Male,Intake officer -&lt;br&gt;Phone call to Compas...,"{'integrated': {'count': 0}, 'presentation': {..."
3,1688893,2179135.0,2272362.0,2272362,42169230.0,642955.0,Progress Note,01/10/2019,08:00,INNER NORTH BRISBANE COMMUNITY MHS,3301188,642955.0,WEN,Johnson,Pindari Mens Hostel 28 Quarry Street Spring Hi...,27/10/1999,Male,&lt;br&gt;Phone call received from&amp;nbsp; P...,"{'integrated': {'count': 0}, 'presentation': {..."
4,1688894,2179137.0,2272365.0,2272365,42169295.0,656929.0,Progress Note,01/10/2019,10:00,INNER NORTH BRISBANE COMMUNITY MHS,NaN,656929.0,WILLS,Tammy,58 Olsen Circuit Kallangur QLD 4503,29/12/1988,Female,1.10.19&lt;br&gt;1000hrs&lt;br&gt;&amp;nbsp;&l...,"{'integrated': {'count': 0}, 'presentation': {..."


In [16]:
from collections import Counter
Counter(list(df['encounterFacility']))

Counter({'GOLD COAST UNIVERSITY HOSPITAL': 324420,
         'INNER NORTH BRISBANE COMMUNITY MHS': 286883,
         "ROYAL BRISBANE &amp;amp; WOMEN'S HOSPITAL": 51767,
         'ROBINA HOSPITAL': 14375,
         'PRINCE CHARLES (THE) HOSPITAL': 10289,
         'Southport Adult Community MHS': 9049,
         'CABOOLTURE HOSPITAL': 8867,
         'REDCLIFFE-CABOOLTURE CRISIS ASSESSMENT &amp;amp; TREATMENT COMMUNITY MHS': 4824,
         'CHERMSIDE ADULT COMMUNITY MHS': 3315,
         'PALM BEACH ADULT COMMUNITY MHS': 3277,
         'Nundah Community Mental Health Service': 2696,
         'COMMUNITY FORENSIC MHS': 2675,
         'REDCLIFFE-CABOOLTURE CHILD &amp;amp; YOUTH COMMUNITY MHS': 2640,
         'REDCLIFFE ADULT COMMUNITY MHS': 2275,
         'PINE RIVERS COMMUNITY MHS': 2107,
         'ROBINA COMMUNITY MHS': 2008,
         'CABOOLTURE ADULT COMMUNITY MHS': 1982,
         'REDCLIFFE-CABOOLTURE COMMUNITY CARE UNIT': 1874,
         'PINE RIVERS COMMUNITY CARE UNIT': 1679,
         'SOU